# XGBoost Split Gain — Worked Example (Exam-Style)

## The Formula You Need to Know Cold

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

Where for each sample $i$: $g_i = \hat{y}_i - y_i$ (gradient), $h_i = \hat{y}_i(1-\hat{y}_i)$ (Hessian), and $G = \sum g_i$, $H = \sum h_i$ over the samples in that node/split.

This is a fill-in-the-numbers exercise once you have $g_i$, $h_i$ per sample. Let's build one from scratch.

---

## Step 1: Toy Dataset (5 transactions, binary fraud label)

| ID | Feature: amount | Label $y$ | Current prediction $\hat{y}$ |
|----|------|-----|------|
| 1 | $50 | 0 | 0.5 |
| 2 | $80 | 0 | 0.5 |
| 3 | $200 | 1 | 0.5 |
| 4 | $350 | 1 | 0.5 |
| 5 | $500 | 1 | 0.5 |

At round 0, every model starts with $\hat{y}=0.5$ for all samples (log-odds = 0).

## Step 2: Compute $g_i$ and $h_i$ for each sample

Using $g_i = \hat{y}_i - y_i$ and $h_i = \hat{y}_i(1-\hat{y}_i)$:

| ID | $y$ | $\hat{y}$ | $g_i = \hat{y}-y$ | $h_i = \hat{y}(1-\hat{y})$ |
|----|-----|-----|------|------|
| 1 | 0 | 0.5 | **0.5** | **0.25** |
| 2 | 0 | 0.5 | **0.5** | **0.25** |
| 3 | 1 | 0.5 | **-0.5** | **0.25** |
| 4 | 1 | 0.5 | **-0.5** | **0.25** |
| 5 | 1 | 0.5 | **-0.5** | **0.25** |

Notice: at $\hat{y}=0.5$ exactly, every Hessian is $0.25$ — that's the max value of $\hat{y}(1-\hat{y})$, a useful sanity check to remember.

## Step 3: Propose a Split — "amount < 150"

- **Left node** (amount < 150): IDs 1, 2
- **Right node** (amount ≥ 150): IDs 3, 4, 5

Sum the gradients and Hessians in each side:

$$G_L = 0.5 + 0.5 = 1.0 \qquad H_L = 0.25+0.25 = 0.5$$
$$G_R = -0.5-0.5-0.5 = -1.5 \qquad H_R = 0.25\times3 = 0.75$$

Parent totals: $G = G_L+G_R = -0.5$, $H = H_L+H_R = 1.25$

## Step 4: Plug into the Gain Formula

Assume regularization $\lambda = 1$, $\gamma = 0.1$ (typical exam-given values).

$$\text{Left term} = \frac{G_L^2}{H_L+\lambda} = \frac{1.0^2}{0.5+1} = \frac{1.0}{1.5} = 0.667$$

$$\text{Right term} = \frac{G_R^2}{H_R+\lambda} = \frac{(-1.5)^2}{0.75+1} = \frac{2.25}{1.75} = 1.286$$

$$\text{Parent term} = \frac{(G_L+G_R)^2}{H_L+H_R+\lambda} = \frac{(-0.5)^2}{1.25+1} = \frac{0.25}{2.25} = 0.111$$

$$\text{Gain} = \frac{1}{2}\big[0.667 + 1.286 - 0.111\big] - 0.1 = \frac{1}{2}(1.842) - 0.1 = 0.921 - 0.1 = \boxed{0.821}$$

Since Gain > 0, this split is **worth making** — it reduces the regularized loss.

## Step 5: Compute the Optimal Leaf Weights (what the leaf actually predicts)

This is the other half examiners like to test — derived from the same second-order Taylor expansion:

$$w^* = -\frac{G}{H+\lambda}$$

$$w_L^* = -\frac{1.0}{0.5+1} = -0.667 \qquad w_R^* = -\frac{-1.5}{0.75+1} = 0.857$$

Interpretation: the left leaf (small amounts, mostly non-fraud) gets pushed toward a **negative** log-odds adjustment (lower fraud probability); the right leaf (larger amounts, mostly fraud) gets pushed **positive** (higher fraud probability). This matches intuition — good exam-answer sanity check.

## Step 6: Convert Leaf Weight Back to Probability (if asked)

New log-odds for right leaf: $0 + \eta \cdot w_R^*$ where $\eta$ is the learning rate (say 0.3):

$$\text{new log-odds} = 0 + 0.3(0.857) = 0.257$$
$$\hat{y}_{new} = \sigma(0.257) = \frac{1}{1+e^{-0.257}} \approx 0.564$$

Fraud probability nudges up from 0.5 → 0.564 after one boosting round — small but directionally correct, as expected with shrinkage.

---

## Exam Checklist — Things to Have Memorized

| Concept | Formula |
|---|---|
| Gradient (logloss) | $g_i = \hat{y}_i - y_i$ |
| Hessian (logloss) | $h_i = \hat{y}_i(1-\hat{y}_i)$ |
| Optimal leaf weight | $w^* = -G/(H+\lambda)$ |
| Optimal leaf objective value | $-\frac{1}{2}\frac{G^2}{H+\lambda}$ |
| Split gain | $\frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right]-\gamma$ |
| Why $\lambda$ matters | Shrinks $w^*$ toward 0, prevents overfit on small-$H$ (few-sample) leaves |
| Why $\gamma$ matters | Minimum gain threshold — prunes splits that don't justify added complexity |

A good exam trap to watch for: they'll sometimes give you $\hat{y}$ values that **aren't** 0.5 (i.e., mid-training, after a few boosting rounds already happened) — same formulas apply, just recompute $g_i, h_i$ from whatever $\hat{y}_i$ is current at that round, not from scratch.

Want me to build a second round example — showing how $\hat{y}$ updates propagate into round-2 gradients, since that's a common follow-up exam question testing whether you understand boosting as *sequential* residual-fitting rather than a one-shot calculation?

#
---

## Relationship between Optimal leaf objective value & Split Gain:

This formula represents the **maximum reduction in loss** (the "Gain") that a single leaf can provide. 

It is not the weight ($w$) itself, but rather the **value of the objective function** after you have plugged the optimal weight $w = -\frac{G}{H+\lambda}$ back into the quadratic approximation.

Here is the mathematical derivation to show how we get from the objective function to this specific value.

### 1. The Objective Function (Simplified)
As established previously, the objective function for a single leaf (ignoring the constant $L(\theta)$ and the complexity penalty $\gamma$) is:
$$Obj(w) = G \cdot w + \frac{1}{2} (H + \lambda) w^2$$
*(Note: I have simplified the notation: $G$ is the sum of gradients, $H$ is the sum of hessians, and $\lambda$ is the regularization.)*

### 2. The Substitution
We know from the previous derivation that the **optimal weight** $w^*$ that minimizes this function is:
$$w^* = -\frac{G}{H + \lambda}$$

To find the **optimal objective value**, we substitute $w^*$ back into the $Obj(w)$ equation:

$$Obj(w^*) = G \left( -\frac{G}{H + \lambda} \right) + \frac{1}{2} (H + \lambda) \left( -\frac{G}{H + \lambda} \right)^2$$

### 3. The Algebra
Let's simplify the second term:
$$Obj(w^*) = -\frac{G^2}{H + \lambda} + \frac{1}{2} (H + \lambda) \left( \frac{G^2}{(H + \lambda)^2} \right)$$

Cancel out one $(H + \lambda)$ in the second term:
$$Obj(w^*) = -\frac{G^2}{H + \lambda} + \frac{1}{2} \frac{G^2}{H + \lambda}$$

Combine the terms ($-1 + \frac{1}{2} = -\frac{1}{2}$):
$$Obj(w^*) = -\frac{1}{2} \frac{G^2}{H + \lambda}$$

---

### **What does this value actually mean?**

In Gradient Boosting, the goal is to **minimize** the loss. 
*   A **negative** value for the objective function means the model has **reduced the loss**.
*   The **larger the magnitude** of this negative number (e.g., $-10$ vs $-2$), the more "useful" that leaf is, because it represents a greater reduction in the error.

**In the context of building a tree:**
When the algorithm is deciding where to split a node, it calculates this value for the potential left child and the potential right child. It then calculates the **Gain**:
$$\text{Gain} = \text{Obj}_{\text{parent}} - (\text{Obj}_{\text{left}} + \text{Obj}_{\text{right}})$$
The algorithm chooses the split that results in the largest positive Gain (the biggest drop in loss).